# RL for Language Models: From PPO to GRPO


The previous three notebooks built the RL toolkit from first principles. [NB1](./01-rl-foundations.html) established the MDP formalism, Bellman equations, and Q-learning. [NB2](./02-policy-gradients.html) developed REINFORCE, baselines, and the actor-critic framework. [NB3](./03-ppo.html) introduced GAE, trust regions, and a full PPO implementation. This notebook applies that entire toolkit to the domain where it matters most today: aligning language models.

LLM alignment is an RL problem. We have a **policy** (the language model), a **reward** (human preferences, answer correctness, safety constraints), and a training loop that must improve the policy without destabilizing it. The connection is not metaphorical — the same equations that govern CartPole govern instruction-following in a 70B parameter model.

We proceed as follows: we cast language generation as an MDP, derive the KL-regularized RLHF objective and its closed-form optimal policy, show how DPO exploits this closed form to eliminate the RL loop entirely, walk through the anatomy of PPO-for-LLMs, and derive GRPO as a principled simplification that drops the value network. We close with a toy reward overoptimization simulation and a unified view of the alignment landscape.

**NOTE:** Implementations of DPO and GRPO live in `tutorial_12_dpo` and `tutorial_14_grpo`. The goal here is the RL-theoretic "why" that makes those implementations deeper.


## 1. Language Generation as an MDP


To apply RL to language models, we must first specify the MDP. There are two natural levels of abstraction: the **token-level MDP**, which models each generation step as a separate decision, and the **bandit (sequence-level) formulation**, which treats the entire response as a single composite action.

**Token-level MDP.** At each step $t$, the agent observes the current state, chooses an action, and transitions deterministically:

- **State**: $s_t = (x, y_{<t})$ — the prompt $x$ concatenated with all tokens generated so far $y_{<t}$.
- **Action**: $a_t = y_t \in \mathcal{V}$ — the next token drawn from vocabulary $\mathcal{V}$.
- **Transition**: deterministic — appending $y_t$ to $(x, y_{<t})$ gives $s_{t+1} = (x, y_{\leq t}).$
- **Reward**: $r_t = 0$ at all intermediate steps; $r_T = r(x, y)$ at the final token, provided by a reward model or human annotator.
- **Policy**: $\pi_\theta(a_t \mid s_t) = p_\theta(y_t \mid x, y_{<t})$ — exactly the language model's next-token distribution.
- **Discount**: $\gamma = 1$, since we care equally about all tokens in a response.

**Bandit (sequence-level) formulation.** When the reward is purely outcome-based — as in math problem solving or preference learning — it is natural to treat generation as a one-step decision:

- **State**: $s = x$ (just the prompt).
- **Action**: $a = y$ (the full response, treated as one composite action).
- **Reward**: $r(x, y)$ at the end.

This is a **contextual bandit** — there is no sequential state transition, only a context-dependent reward for each response. GRPO operates at this level. PPO-for-LLMs operates at the token level.

<br>

The correspondence with standard RL is exact:

| RL Concept | LLM Equivalent |
|---|---|
| State $s_t$ | Prompt + generated tokens so far $(x, y_{<t})$ |
| Action $a_t$ | Next token $y_t \in \mathcal{V}$ |
| Policy $\pi_\theta(a \mid s)$ | Language model $p_\theta(y_t \mid x, y_{<t})$ |
| Trajectory $\tau$ | (prompt, response) pair $(x, y)$ |
| Return $G_0$ | Reward $r(x, y)$ |
| Episode length $T$ | Response length $|y|$ |
| Environment | Human preferences / reward model |

: {tbl-colwidths="[30,70]"}


:::{.callout-note}
The "action space" in LLM RL is the vocabulary — typically 32K–128K tokens. For comparison, CartPole has 2 actions and Atari games have 18. This scale makes value-based methods (Q-learning from NB1) completely intractable and motivates policy gradient approaches.

:::


## 2. The RLHF Objective


We want to train a policy that maximizes reward while staying close to the reference model $\pi_\text{ref}$ (typically the SFT-finetuned model). The unconstrained reward maximization objective is degenerate — the policy would collapse to a degenerate distribution over the single highest-reward response, sacrificing all diversity and exploiting reward model imperfections. The standard fix is a **KL-regularized objective**:

$$\max_{\pi_\theta} \; \mathbb{E}_{x \sim \mathcal{D},\; y \sim \pi_\theta(\cdot \mid x)}\left[r(x, y)\right] - \beta \, D_\text{KL}\!\left(\pi_\theta(\cdot \mid x) \,\|\, \pi_\text{ref}(\cdot \mid x)\right)$$

The parameter $\beta > 0$ controls the tradeoff: large $\beta$ keeps $\pi_\theta$ close to $\pi_\text{ref}$; small $\beta$ allows the policy to pursue reward aggressively.

**Connection to NB3.** The KL constraint here is exactly the trust region from TRPO. Instead of the hard KL constraint $D_\text{KL} \leq \delta$, RLHF uses a soft KL penalty. The intuition is identical: prevent the policy from drifting too far from the known-good starting point.

<br>

**Deriving the optimal policy.** This is the key theoretical result. We derive the closed-form solution to the RLHF objective step by step.

Expanding the KL divergence, the objective can be written as:

$$J(\pi_\theta) = \mathbb{E}_{x, y}\left[r(x,y) - \beta \log \frac{\pi_\theta(y \mid x)}{\pi_\text{ref}(y \mid x)}\right]$$

For a fixed prompt $x$, we maximize over the distribution $\pi_\theta(\cdot \mid x)$:

$$\sum_y \pi_\theta(y \mid x)\left[r(x,y) - \beta \log \frac{\pi_\theta(y \mid x)}{\pi_\text{ref}(y \mid x)}\right]$$

subject to $\sum_y \pi_\theta(y \mid x) = 1.$ Taking the functional derivative with respect to $\pi_\theta(y \mid x)$ and setting it to zero (with a Lagrange multiplier for the normalization constraint), we obtain:

$$\boxed{\pi^*(y \mid x) = \frac{1}{Z(x)}\, \pi_\text{ref}(y \mid x)\, \exp\!\left(\frac{r(x,y)}{\beta}\right)}$$

where the **partition function** $Z(x) = \sum_{y'} \pi_\text{ref}(y' \mid x) \exp(r(x,y')/\beta)$ normalizes the distribution.

The optimal policy $\pi^*$ upweights responses with high reward by a factor $\exp(r/\beta)$ relative to the reference, then renormalizes. This is a **Boltzmann distribution** over responses — the same soft-max structure that appears in the softmax Bellman equation from NB1. Responses far from the reference require strong reward signal to be preferred; responses near the reference are favored by default.


:::{.callout-important}
The partition function $Z(x)$ is intractable — summing over all possible responses is exponential in the response length. This is why we cannot directly compute $\pi^*$; we need approximate optimization methods (PPO, GRPO) or a clever reparameterization (DPO).

:::


## 3. DPO as Implicit RL


DPO's key insight is that the intractable partition function $Z(x)$ can be made to vanish. Starting from the optimal policy equation and solving for the reward:

$$r(x,y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_\text{ref}(y \mid x)} + \beta \log Z(x)$$

The **Bradley-Terry preference model** gives the probability that response $y_w$ is preferred over $y_l$:

$$p(y_w \succ y_l \mid x) = \sigma\!\left(r(x,y_w) - r(x,y_l)\right)$$

Crucially, $\beta \log Z(x)$ cancels in the difference $r(x,y_w) - r(x,y_l)$. Substituting the parameterization $r(x,y) \approx \beta \log \frac{\pi_\theta(y \mid x)}{\pi_\text{ref}(y \mid x)}$ and maximizing the log-likelihood over a preference dataset of $(x, y_w, y_l)$ triples:

$$\mathcal{L}_\text{DPO}(\theta) = -\mathbb{E}_{(x, y_w, y_l)}\!\left[\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_\text{ref}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_\text{ref}(y_l \mid x)}\right)\right]$$

**The RL interpretation.** DPO trains $\pi_\theta$ to be the optimal policy $\pi^*$ of the RLHF objective, without ever running an RL loop. The log-ratio $\beta \log \pi_\theta / \pi_\text{ref}$ is the **implicit reward** — DPO simultaneously acts as a reward model and a policy optimizer.

<br>

**Tradeoffs.** The two families of alignment methods have complementary strengths:

- **DPO**: stable, no reward model needed, no RL loop — but offline; it cannot improve beyond what the fixed preference dataset implies.
- **PPO / GRPO**: online — the policy generates its own rollouts and can discover new high-reward behaviors — but requires an explicit reward model and a more complex optimization loop.

For the full DPO derivation and implementation, see `tutorial_12_dpo`.


## 4. PPO for Language Models


PPO-for-LLMs is the same algorithm from NB3, adapted for the token-level LLM MDP. The clipped surrogate objective $L^\text{CLIP}$, importance ratios $\rho = \pi_\theta / \pi_{\theta_\text{old}}$, GAE for advantage estimation, and multiple epochs of minibatch updates are all unchanged. What changes is the environment and the reward structure.

**What changes for LLMs:**

1. **Four models in memory**: policy $\pi_\theta$, reference $\pi_\text{ref}$, value model $V_\phi$, reward model $r_\psi$. The reference model must be kept frozen and queried every rollout.
2. **KL penalty added to reward**: the per-token shaped reward subtracts a KL term at every step, and adds the outcome reward only at the final token:
$$\tilde{r}_t = -\beta \left(\log \pi_\theta(y_t \mid s_t) - \log \pi_\text{ref}(y_t \mid s_t)\right) + r(x,y) \cdot \mathbf{1}[t = T]$$
3. **Value network on the LM backbone**: the value head must share or duplicate the LM's weights, roughly doubling memory.
4. **Sparse reward**: the outcome reward arrives only at the final token, so GAE is essential for propagating credit back through the token sequence.

<br>

**Annotated pseudocode.** The following summarizes the PPO-for-LLMs training loop:


**PPO-for-LLMs pseudocode** — illustrating the four-model setup and KL-shaped reward:


In [ ]:
# PPO-for-LLMs pseudocode (not executable — illustrative)
# 4 models: policy, reference, value_model, reward_model

for batch in dataloader:
    prompts = batch["prompts"]                              # <1>

    # == ROLLOUT PHASE ==
    responses = policy.generate(prompts)                    # <2>
    rewards = reward_model.score(prompts, responses)        # <3>
    ref_logprobs = reference.logprobs(prompts, responses)   # <4>
    old_logprobs = policy.logprobs(prompts, responses)      # <5>
    values = value_model(prompts, responses)                # <6>

    # KL-shaped reward (per token)
    kl = old_logprobs - ref_logprobs                        # <7>
    shaped_rewards = -beta * kl                             # <8>
    shaped_rewards[:, -1] += rewards                        # <9>

    # GAE advantages (same as NB3!)
    advantages, returns = compute_gae(
        shaped_rewards, values, gamma=1.0, lam=0.95         # <10>
    )

    # == OPTIMIZATION PHASE (K epochs) ==
    for epoch in range(K):
        for mb in minibatches(prompts, responses, ...):
            new_logprobs = policy.logprobs(mb.prompts, mb.responses)
            ratio = (new_logprobs - mb.old_logprobs).exp()  # <11>

            surr = torch.min(
                ratio * mb.advantages,
                ratio.clamp(1-eps, 1+eps) * mb.advantages  # <12>
            )
            policy_loss = -surr.mean()
            value_loss = mse(value_model(...), mb.returns)
            loss = policy_loss + c1 * value_loss

            loss.backward(); optimizer.step()


1. Each batch contains a set of prompts drawn from the training distribution $\mathcal{D}.$
2. The policy generates responses autoregressively — this is the online sampling step that PPO requires.
3. The reward model scores each (prompt, response) pair, returning a scalar outcome reward.
4. The frozen reference model computes log-probabilities of the generated responses — needed for the per-token KL.
5. The current policy's log-probabilities are recorded before optimization; these become $\pi_{\theta_\text{old}}$ for the importance ratio.
6. The value model predicts $V(s_t)$ at each token position, providing the baseline for GAE.
7. Per-token KL divergence: $\log \pi_{\theta_\text{old}}(y_t \mid s_t) - \log \pi_\text{ref}(y_t \mid s_t).$ Positive when the policy has moved away from the reference.
8. The shaped per-token reward is $-\beta \cdot \text{KL}_t$ — a continuous penalty that discourages drift from the reference at every step.
9. The outcome reward is added only at the final token position, matching the sparse reward structure of the MDP.
10. GAE propagates the sparse terminal reward backward through the sequence, producing per-token advantages. $\gamma = 1$ treats all tokens equally; $\lambda = 0.95$ trades bias for variance.
11. Importance ratio $\rho_t = \pi_\theta(y_t \mid s_t) / \pi_{\theta_\text{old}}(y_t \mid s_t)$, computed in log space for numerical stability.
12. The PPO clip (from NB3) prevents large policy updates: if $\rho_t$ moves outside $[1-\epsilon, 1+\epsilon]$, the gradient is zeroed.


**Memory cost.** For a 7B model in `bfloat16`, each model requires approximately 14 GB. Four models — policy, reference, value, and reward — amount to roughly 56 GB just for weights, before activations or optimizer states. In practice the value model shares the LM backbone with the policy (adding only a small head), but the reference and reward models are fully separate. This cost motivates GRPO, which eliminates the value network entirely.


## 5. GRPO — PPO Without the Value Network


GRPO (Group Relative Policy Optimization) was introduced in DeepSeekMath [@shao2024deepseekmath]. It eliminates the value network by replacing the learned $V_\phi$ baseline with a Monte Carlo estimate computed from a group of sampled responses.

**The key idea.** For each prompt $x$, we sample a group of $G$ responses $y_1, \ldots, y_G \sim \pi_\theta(\cdot \mid x)$ and score each with the reward model. The group mean reward is:

$$\hat{V}^\pi(x) \approx \bar{r} = \frac{1}{G}\sum_{i=1}^G r(x, y_i)$$

The normalized group-relative advantage for response $i$ is then:

$$\hat{A}_i = \frac{r_i - \bar{r}}{\sigma_r}, \quad \sigma_r = \text{std}(r_1, \ldots, r_G)$$

**Why is this a valid baseline?** From NB2: any baseline $b(s)$ that depends only on the state and not the action yields an unbiased policy gradient estimator. The group mean $\bar{r}$ depends on the prompt $x$ (the "state") and is marginalized over responses — it is a Monte Carlo estimate of $V^\pi(x).$ Technically, $\bar{r}$ includes response $y_i$ itself, introducing a small bias; but with $G > 1$ this is the REINFORCE-leave-one-out estimator, which is consistent.

<br>

**Component comparison.** GRPO and PPO-for-LLMs solve the same problem with different approximations:

| Component | PPO-for-LLMs | GRPO | Why GRPO works |
|---|---|---|---|
| Advantage | GAE via value network | $(r_i - \bar{r}) / \sigma_r$ | Group mean $\approx V^\pi(x)$ |
| Baseline | $V_\phi(s_t)$ per token | $\bar{r}$ per sequence | MC baseline (NB2) |
| Importance ratio | Per-token | Per-token | Both use PPO clipping |
| Models | 4 (policy, ref, value, reward) | 3 (policy, ref, reward) | No value network |
| Memory | $\sim 4\times$ LM | $\sim 3\times$ LM | 25% reduction |

: {tbl-colwidths="[22,26,26,26]"}

<br>

**The GRPO loss.** For a group of $G$ responses to prompt $x$, the GRPO objective is:

$$L^{\text{GRPO}}(\theta) = -\frac{1}{G}\sum_{i=1}^G \frac{1}{|y_i|}\sum_{t=1}^{|y_i|} \left[\min\!\left(\rho_{i,t}\hat{A}_i,\; \text{clip}(\rho_{i,t}, 1-\epsilon, 1+\epsilon)\hat{A}_i\right) - \beta\, D_\text{KL}(\pi_\theta \| \pi_\text{ref})\right]$$

where $\rho_{i,t} = \pi_\theta(y_{i,t} \mid x, y_{i,<t}) / \pi_{\theta_\text{old}}(y_{i,t} \mid x, y_{i,<t})$ is the per-token importance ratio. The clipping and KL penalty are inherited directly from PPO-for-LLMs. What GRPO replaces is only the source of the advantage estimate.


:::{.callout-note}
For the full GRPO implementation — sampling groups of responses, scoring, computing the loss, training loop, and monitoring reward hacking — see `tutorial_14_grpo`. The end-to-end pipeline (SFT → reward model → GRPO) is in `tutorial_17_capstone`.

:::


## 6. Reward Hacking and the KL Constraint


**Reward hacking** (Goodhart's Law applied to RL) is the phenomenon where a policy over-optimizes a proxy reward at the expense of true performance. It is a fundamental failure mode of any learned reward signal, and it becomes acute in LLM alignment because reward models are trained on finite human preference data and are therefore imperfect.

The mechanism is straightforward:

1. The reward model is trained on a finite preference dataset and fits the training distribution well.
2. As the policy moves far from that distribution, the reward model's predictions become unreliable — it extrapolates badly.
3. The policy discovers "exploits" — high-proxy-reward, low-true-reward responses — by gradient descent.

The KL penalty $\beta \, D_\text{KL}(\pi_\theta \| \pi_\text{ref})$ is a trust region that limits how far the policy can drift from the reference. From NB3, this is exactly TRPO's KL constraint applied as a continuous soft penalty. The $\beta$ parameter governs the tradeoff between reward optimization and fidelity to the reference.

We demonstrate this with a minimal simulation. The "true reward" is a Gaussian bump peaked at $x = 0$; the "proxy reward" is a polynomial approximation that matches near the origin but diverges for large $|x|.$ The policy is a Gaussian with mean $\mu$ — training moves $\mu$ toward higher proxy reward. We compare outcomes with $\beta \in \{0, 0.1, 0.5\}.$


**Setup.** We import NumPy and configure SVG output:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")


**Reward overoptimization simulation.** We define the true and proxy rewards, then simulate policy gradient ascent on the proxy under varying KL penalties:


In [ ]:
def true_reward(x):
    """True reward: Gaussian bump centered at 0."""
    return np.exp(-x**2 / 2)

def proxy_reward(x):
    """Proxy reward: polynomial that matches true reward near 0 but diverges for large |x|."""
    return 1 - 0.5 * x**2 + 0.05 * np.abs(x)**3  # <1>

def simulate_overoptimization(n_steps=300, lr=0.03, beta=0.0, sigma=0.5):
    """Gradient ascent on proxy reward with optional KL penalty."""
    mu = 0.0      # policy mean starts at reference
    mu_ref = 0.0  # reference policy mean
    history = {"step": [], "mu": [], "proxy": [], "true": [], "kl": []}

    for step in range(n_steps):
        kl = (mu - mu_ref)**2 / (2 * sigma**2)       # <2>
        grad_proxy = -mu                               # <3>
        grad_kl = (mu - mu_ref) / sigma**2
        mu += lr * (grad_proxy - beta * grad_kl)       # <4>

        history["step"].append(step)
        history["mu"].append(mu)
        history["proxy"].append(proxy_reward(mu))
        history["true"].append(true_reward(mu))
        history["kl"].append(kl)

    return history


1. The proxy reward is a degree-3 polynomial. Near $x = 0$ it closely approximates the true Gaussian reward, but for $|x| \gtrsim 2$ the cubic term dominates and the proxy diverges — a simple model of reward misspecification.
2. For two Gaussians with equal variance $\sigma^2$, the KL divergence reduces to $(\mu - \mu_\text{ref})^2 / (2\sigma^2).$ This is zero when the policy matches the reference and grows quadratically with drift.
3. The gradient of the proxy reward evaluated at the policy mean — the policy gradient pushes $\mu$ toward the proxy maximum.
4. The update adds the proxy gradient and subtracts a term proportional to the KL gradient, acting as a quadratic regularizer centered on $\mu_\text{ref}.$


**Visualization.** We run the simulation for three values of $\beta$ and plot the reward trajectories together with the reward landscape:


In [ ]:
#| code-fold: true
betas = [0.0, 0.1, 0.5]
colors = ["#e74c3c", "#2ecc71", "#3498db"]
labels = [r"$\beta = 0$ (no KL)", r"$\beta = 0.1$", r"$\beta = 0.5$"]
histories = [simulate_overoptimization(beta=b) for b in betas]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# --- Left panel: reward curves over training steps ---
ax = axes[0]
for h, c, lab in zip(histories, colors, labels):
    ax.plot(h["step"], h["true"], color=c, linewidth=1.8, label=lab)

ax.axhline(1.0, color="gray", linestyle="dashed", linewidth=1.0, alpha=0.6,
           label="true optimum")
ax.set_xlabel("training step", fontsize=11)
ax.set_ylabel("true reward", fontsize=11)
ax.set_title("True reward during training", fontsize=12)
ax.legend(fontsize=9)
ax.grid(linestyle="dotted", alpha=0.5)
ax.set_xlim(0, 300)

# --- Right panel: reward landscape with final policy positions ---
ax = axes[1]
xs = np.linspace(-4, 4, 500)
ax.plot(xs, true_reward(xs), "k-",  linewidth=2.0, label="true reward", zorder=3)
ax.plot(xs, proxy_reward(xs), "k--", linewidth=1.5, alpha=0.5, label="proxy reward", zorder=3)
ax.fill_between(xs, true_reward(xs), proxy_reward(xs),
                where=(proxy_reward(xs) > true_reward(xs)),
                alpha=0.08, color="red", label="overoptimization region")

for h, c, lab in zip(histories, colors, labels):
    final_mu = h["mu"][-1]
    ax.axvline(final_mu, color=c, linewidth=1.5, alpha=0.8, linestyle="-")
    ax.scatter([final_mu], [true_reward(final_mu)], color=c, s=60, zorder=5)

ax.set_xlabel(r"policy mean $\mu$", fontsize=11)
ax.set_ylabel("reward", fontsize=11)
ax.set_title("Reward landscape and final policy positions", fontsize=12)
ax.legend(fontsize=9)
ax.grid(linestyle="dotted", alpha=0.5)
ax.set_xlim(-4, 4)
ax.set_ylim(-0.3, 1.4)

plt.tight_layout()
plt.show()


**Figure.** (**Left**) True reward achieved during training for each $\beta.$ Without KL regularization ($\beta = 0$, red), the policy overshoots the true reward peak and settles at a suboptimal region where the proxy is high but the true reward is low. (**Right**) The reward landscape showing the divergence between true and proxy rewards for $|x| > 2,$ together with the final policy mean $\mu$ for each $\beta.$ The $\beta = 0$ policy overshoots into the overoptimization region; $\beta = 0.1$ lands near the true optimum; $\beta = 0.5$ barely moves from the reference.

The $\beta$ parameter in DPO, GRPO, and PPO-for-LLMs controls exactly this tradeoff in the actual alignment setting. Practitioners typically sweep $\beta$ or schedule it to allow aggressive early optimization while tightening the constraint later.


## 7. The Alignment Landscape


We can now place all major alignment methods on two axes: (1) whether training is **online** — the policy generates its own data during training — or **offline** — the dataset is fixed; and (2) whether an **explicit reward model** is required.

| Method | Online? | Explicit RM? | RL Algorithm | Reference |
|---|---|---|---|---|
| SFT | Offline | No | Behavioral cloning | `tutorial_11_sft_lora` |
| DPO | Offline | No (implicit) | Implicit RL (closed-form) | `tutorial_12_dpo` |
| GRPO | Online | Yes | REINFORCE + clipping | `tutorial_14_grpo` |
| PPO-for-LLMs | Online | Yes | PPO (full) | — |

: {tbl-colwidths="[22,14,20,28,16]"}

<br>

**The GPI connection.** The full alignment pipeline — SFT → reward model training → GRPO/PPO — is **generalized policy iteration** in disguise. Recall from NB1 that GPI alternates between policy evaluation (estimating $V^\pi$) and policy improvement (acting greedily w.r.t. $V^\pi$) until convergence. The alignment pipeline instantiates this at the scale of human preferences:

- **Policy evaluation** step: train the reward model on human-labeled preference data to estimate the "value" of responses.
- **Policy improvement** step: run GRPO or PPO to improve the policy using the current reward model.
- The cycle: a better policy generates higher-quality outputs → better preference data can be collected → a better reward model is trained → the policy improves further.

<br>

**Open directions.** Several active research threads push beyond the methods covered here:

- **Online DPO / Iterative DPO**: generate new preference pairs online from the current policy, then apply DPO on the fresh data. This combines DPO's stability with GRPO's ability to explore beyond the initial dataset.
- **Process Reward Models (PRM)**: instead of a sparse outcome reward at the final token, PRMs provide a dense reward at each reasoning step. This directly addresses the credit assignment problem and is essential for long-horizon chain-of-thought tasks.
- **Test-time compute scaling**: using RL methods at inference time — best-of-$N$ sampling, MCTS with a value function as a search heuristic. The trained value network becomes a search guide rather than just a baseline.
- **Constitutional AI / RLAIF**: the reward model is itself a language model, eliminating the need for human labels at scale. Feedback is generated by asking a "constitution"-following LM to rate responses, enabling self-improving alignment loops.


---


## Appendix: DPO from the Soft Bellman Equation {#sec-appendix-dpo-bellman}


This appendix is for readers who want the deepest unification. We show that the DPO implicit reward $r(x,y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_\text{ref}(y \mid x)}$ is precisely the **advantage function** of the KL-constrained MDP, and that DPO is therefore training a policy to match the soft Bellman optimum.

**MaxEnt RL and the soft Bellman equation.** The KL-regularized RLHF objective belongs to the family of **maximum entropy RL** (MaxEnt RL), also studied in the Soft Actor-Critic (SAC) algorithm. In the bandit formulation, the Q-function is simply the reward: $Q^*(x,y) = r(x,y).$ The **soft-optimal policy** satisfies:

$$\pi^*(y \mid x) \propto \pi_\text{ref}(y \mid x) \exp\!\left(\frac{Q^*(x,y)}{\beta}\right)$$

which is exactly the optimal policy we derived in §2. The corresponding **soft value function** is:

$$V^*(x) = \beta \log \sum_y \pi_\text{ref}(y \mid x) \exp\!\left(\frac{r(x,y)}{\beta}\right) = \beta \log Z(x)$$

The **advantage function** measures how much better a particular response $y$ is compared to the average:

$$\begin{aligned}
A^*(x,y) &= Q^*(x,y) - V^*(x) \\[0.5em]
&= r(x,y) - \beta \log Z(x) \\[0.5em]
&= \beta \log \frac{\pi^*(y \mid x)}{\pi_\text{ref}(y \mid x)}
\end{aligned}$$

The last equality follows directly from the optimal policy formula: $\pi^*(y \mid x) = \frac{1}{Z(x)} \pi_\text{ref}(y \mid x) \exp(r/\beta).$ The partition function $Z(x)$ cancels when computing $A^*.$

**DPO as advantage matching.** DPO parameterizes $A^*(x,y) \approx \beta \log \frac{\pi_\theta(y \mid x)}{\pi_\text{ref}(y \mid x)}$ and trains $\pi_\theta$ to reproduce the **rankings** induced by $A^*$ — which is precisely the Bradley-Terry preference model. The preference probability becomes:

$$p(y_w \succ y_l \mid x) = \sigma\!\left(A^*(x,y_w) - A^*(x,y_l)\right) = \sigma\!\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_\text{ref}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_\text{ref}(y_l \mid x)}\right)$$

Maximizing the log-likelihood of observed preferences over $\theta$ is exactly $-\mathcal{L}_\text{DPO}(\theta).$

<br>

**The unified picture.** DPO, GRPO, and PPO-for-LLMs are all optimizing the same soft Bellman equation for the KL-constrained MDP, with different approximations:

- **DPO**: closed-form, offline. The advantage is computed exactly from the policy ratio; no sampling required. The price: fixed to the offline preference dataset.
- **GRPO**: online, sequence-level. The advantage $\hat{A}_i = (r_i - \bar{r})/\sigma_r$ is a Monte Carlo estimate of $A^\pi(x, y_i)$ using the group mean as $V^\pi(x).$ No value network needed.
- **PPO-for-LLMs**: online, token-level. The advantage is computed via GAE using a learned value network $V_\phi \approx V^\pi.$ Higher variance reduction, higher memory cost.

All three share the same objective; they differ in how they estimate the gradient of that objective.


---


■
